In [ ]:
import shap
import torch
from torch import nn
from PIL import Image
import numpy as np
import os, copy
import math
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import auc
from skimage.transform import resize
from transformers import AutoImageProcessor, ViTModel
from transformers import RobertaTokenizer, RobertaModel
from scipy.ndimage import gaussian_filter, convolve #same library as rise paper uses
import textwrap
from matplotlib.colors import Normalize
from scipy.stats import spearmanr
import cv2
import random
from sklearn.linear_model import Ridge
from sklearn.metrics.pairwise import cosine_distances
from PIL import Image, ImageDraw
import matplotlib.colors as mcolors

In [ ]:
use_cuda = False#True
RANDOM_STATE = 8
IMAGE_MODEL = 'google/vit-base-patch16-384'
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
combined_df = pd.read_csv('../dataset.csv')
combined_df = combined_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
train_df, test_df = train_test_split(combined_df, test_size=0.2, random_state=RANDOM_STATE, stratify=combined_df['label1'])
test_df = test_df.reset_index(drop=True)
train_df = train_df.reset_index(drop=True)
label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(train_df['voted_label'].values)
test_labels = label_encoder.transform(test_df['voted_label'].values)
test_df[:3]

In [ ]:
image_processor = AutoImageProcessor.from_pretrained(IMAGE_MODEL)
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

In [ ]:
class Model(nn.Module):
    def __init__(self, num_classes=13, embed_dim=512):
        super().__init__()
        self.image_encoder = ViTModel.from_pretrained("google/vit-base-patch16-384")
        self.text_encoder = RobertaModel.from_pretrained("roberta-base")
        # Project image and text features to same dimension
        self.image_proj = nn.Linear(768, embed_dim)
        self.text_proj = nn.Linear(768, embed_dim)
        # Transformer encoder to fuse both modalities
        self.transformer_fusion = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, dim_feedforward=2048, dropout=0.2), 
            num_layers=2)
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, image, output_attentions=False):
        # Encode image and text
        image_out = self.image_encoder(image, output_attentions=output_attentions)
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask, output_attentions=output_attentions)
        image_feat = image_out.last_hidden_state[:, 0]
        text_feat = text_out.last_hidden_state[:, 0]
        # Project to same dimensional space
        image_embeds = self.image_proj(image_feat)
        text_embeds = self.text_proj(text_feat)
        # Prepare for Transformer
        fused_input = torch.stack([image_embeds, text_embeds], dim=0)
        # Fuse with Transformer
        fused_output = self.transformer_fusion(fused_input)
        # Average fused output across image/text tokens
        fused_embeds = fused_output.mean(dim=0) #fused_output
        # Classification
        logits = self.classifier(fused_embeds)
        if output_attentions:
            return logits, text_out.attentions, image_out.attentions  # Return both logits and attention maps for text & image
        return logits

def query_model(image_input, text_input):
    model.eval()
    with torch.no_grad():
        image_input = image_input.to(DEVICE)
        input_ids = text_input['input_ids'].to(DEVICE)
        attention_mask = text_input['attention_mask'].to(DEVICE)
        logits = model(input_ids, attention_mask, image_input)
        return logits

def insertion_deletion(image_input, text_input, lbl, sorted_scores, mode, media, return_output_changes=False, masking_type='zero'):
    outputs = query_model(image_input, text_input)
    softmax_output = torch.softmax(outputs, dim=1)
    _, cat = torch.max(softmax_output, -1)
    baseline_class = int(cat.cpu().numpy()[0])
    baseline_prob = softmax_output[0][baseline_class].item()
    baseline_correct_pred = (True)
    baseline_class_str = label_encoder.inverse_transform([baseline_class])[0]
    if baseline_class_str != lbl:
        baseline_correct_pred = (False, lbl, baseline_class_str)
    
    if media == 'image':
        grid_size = 12
        img_patch_size_px = image_input.shape[2] // grid_size
        patched_img = copy.deepcopy(image_input).squeeze().detach().cpu().numpy()
        if masking_type == 'blur': # original rise blurred image masking
            inp = np.zeros((11, 11))
            inp[11//2, 11//2] = 1
            k = gaussian_filter(inp, 5)
            masked_img = np.zeros_like(patched_img)
            for c in range(3):
                masked_img[c] = convolve(patched_img[c], k, mode='constant', cval=0.0)
        if masking_type == 'zero':
            masked_img = np.zeros_like(patched_img)
        if masking_type == 'mean':
            mean_per_channel = patched_img.reshape(3, -1).mean(axis=1)
            masked_img = np.zeros_like(patched_img)
            for c in range(3):
                masked_img[c, :, :] = mean_per_channel[c]
    if media == 'text':
        text_input_copy = copy.deepcopy(text_input)
        txt_input_ids = text_input_copy['input_ids'].squeeze().detach().cpu().numpy()
        if mode == 'ins':
            masked_txt = np.full_like(txt_input_ids, fill_value=50264)
            for i in range(len(masked_txt)):
                if txt_input_ids[i] == 0:
                    masked_txt[i] = 0
                if txt_input_ids[i] == 2:
                    masked_txt[i] = 2

    n_steps = len(sorted_scores)
    scores = np.empty(n_steps)
    output_changes = [] if return_output_changes else None

    for i in range(n_steps):
        if media == 'image':
            img_patch_idx = sorted_scores[i][0]
            row = img_patch_idx // grid_size
            col = img_patch_idx % grid_size
            start_row = row * img_patch_size_px
            end_row = (row + 1) * img_patch_size_px
            start_col = col * img_patch_size_px
            end_col = (col + 1) * img_patch_size_px
            if mode == 'del':
                if masking_type == 'zero':
                    patched_img[:, start_row:end_row, start_col:end_col] = 0
                if masking_type == 'blur':
                    patched_img[:, start_row:end_row, start_col:end_col] = masked_img[:, start_row:end_row, start_col:end_col]
                new_image_input = torch.tensor(patched_img).unsqueeze(0)
            if mode == 'ins':
                masked_img[:, start_row:end_row, start_col:end_col] = patched_img[:, start_row:end_row, start_col:end_col]
                new_image_input = torch.tensor(masked_img).unsqueeze(0)
            outputs = query_model(new_image_input, text_input)
        if media == 'text':
            txt_patch_idx = sorted_scores[i][0]
            if mode == 'del':
                if txt_input_ids[txt_patch_idx] != 0:
                    if txt_input_ids[txt_patch_idx] != 2:
                        txt_input_ids[txt_patch_idx] = 50264
                text_input_copy['input_ids'] = torch.tensor(txt_input_ids).unsqueeze(0)
            if mode == 'ins':
                masked_txt[txt_patch_idx] = txt_input_ids[txt_patch_idx]
                text_input_copy['input_ids'] = torch.tensor(masked_txt).unsqueeze(0)
            outputs = query_model(image_input, text_input_copy)
        softmax_output = torch.softmax(outputs, dim=1)
        prob = softmax_output[0][baseline_class].item() #we want the prob of baseline predicted class rather than the label so we can see later how results differ
        scores[i] = prob
        if return_output_changes:
            if mode == 'del':
                change = abs(baseline_prob - prob)
            elif mode == 'ins':
                change = abs(prob - baseline_prob)
            output_changes.append(change)
    if return_output_changes:
        return scores, baseline_correct_pred, output_changes
    else:
        return scores, baseline_correct_pred

def insertion_deletion_both(image_input, text_input, lbl, sorted_scores_img, sorted_scores_txt, mode, return_output_changes=False, masking_type='zero'):
    # baseline prediction
    outputs = query_model(image_input, text_input)
    softmax_output = torch.softmax(outputs, dim=1)
    _, cat = torch.max(softmax_output, -1)
    baseline_class = int(cat.cpu().numpy()[0])
    baseline_prob = softmax_output[0][baseline_class].item()
    baseline_correct_pred = (True)
    baseline_class_str = label_encoder.inverse_transform([baseline_class])[0]
    if baseline_class_str != lbl:
        baseline_correct_pred = (False, lbl, baseline_class_str)
    
    # create image substrate
    grid_size = 12
    img_patch_size_px = image_input.shape[2] // grid_size
    patched_img = copy.deepcopy(image_input).squeeze().detach().cpu().numpy()
    if masking_type == 'blur': # original rise blurred image masking
        inp = np.zeros((11, 11))
        inp[11//2, 11//2] = 1
        k = gaussian_filter(inp, 5)
        masked_img = np.zeros_like(patched_img)
        for c in range(3):
            masked_img[c] = convolve(patched_img[c], k, mode='constant', cval=0.0)
    if masking_type == 'zero':
        masked_img = np.zeros_like(patched_img)
    if masking_type == 'mean':
        mean_per_channel = patched_img.reshape(3, -1).mean(axis=1)
        masked_img = np.zeros_like(patched_img)
        for c in range(3):
            masked_img[c, :, :] = mean_per_channel[c]

    # create text substrate
    text_input_copy = copy.deepcopy(text_input)
    txt_input_ids = text_input_copy['input_ids'].squeeze().detach().cpu().numpy()
    if mode == 'ins':
        masked_txt = np.full_like(txt_input_ids, fill_value=50264)
        for i in range(len(masked_txt)):
            if txt_input_ids[i] == 0:
                masked_txt[i] = 0
            if txt_input_ids[i] == 2:
                masked_txt[i] = 2

    n_steps = len(sorted_scores_img)
    scores = np.empty(n_steps)
    output_changes = [] if return_output_changes else None

    for i in range(n_steps):
        # image
        img_patch_idx = sorted_scores_img[i][0]
        row = img_patch_idx // grid_size
        col = img_patch_idx % grid_size
        start_row = row * img_patch_size_px
        end_row = (row + 1) * img_patch_size_px
        start_col = col * img_patch_size_px
        end_col = (col + 1) * img_patch_size_px
        if mode == 'del':
            if masking_type == 'zero':
                patched_img[:, start_row:end_row, start_col:end_col] = 0
            if masking_type == 'blur':
                patched_img[:, start_row:end_row, start_col:end_col] = masked_img[:, start_row:end_row, start_col:end_col]
            new_image_input = torch.tensor(patched_img).unsqueeze(0)
        if mode == 'ins':
            masked_img[:, start_row:end_row, start_col:end_col] = patched_img[:, start_row:end_row, start_col:end_col]
            new_image_input = torch.tensor(masked_img).unsqueeze(0)
        
        # text
        txt_patch_idx = sorted_scores_txt[i][0]
        if mode == 'del':
            if txt_input_ids[txt_patch_idx] != 0:
                if txt_input_ids[txt_patch_idx] != 2:
                    txt_input_ids[txt_patch_idx] = 50264
            text_input_copy['input_ids'] = torch.tensor(txt_input_ids).unsqueeze(0)
        if mode == 'ins':
            masked_txt[txt_patch_idx] = txt_input_ids[txt_patch_idx]
            text_input_copy['input_ids'] = torch.tensor(masked_txt).unsqueeze(0)

        outputs = query_model(new_image_input, text_input_copy)
        softmax_output = torch.softmax(outputs, dim=1)
        prob = softmax_output[0][baseline_class].item() #we want the prob of baseline predicted class rather than the label so we can see later how results differ
        scores[i] = prob
        if return_output_changes:
            if mode == 'del': #baseline_prob remains high while prob will start high and decrease - change should increase as features are added ~0:100
                change = abs(baseline_prob - prob)
            elif mode == 'ins': #prob will start low and increase while baseline_prob remains high - change should decrease as features are added ~100:0
                change = abs(prob - baseline_prob)
            output_changes.append(change)
    if return_output_changes:
        return scores, baseline_correct_pred, output_changes
    else:
        return scores, baseline_correct_pred

In [ ]:
num_labels = 13
model = Model(num_labels)
model.to(DEVICE)
model_save_path = '../3foldcv/mmtf_r8'
save_path = os.path.join(model_save_path, "checkpoint.pth")
checkpoint = torch.load(save_path)
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
custom_columns = ['sample_idx', 'shap', 'rise', 'lime','attention', 'rollout', 'random']
results_df = pd.DataFrame(columns=custom_columns)

In [ ]:
# perturbation based
def compute_shap_auc_scores(image_input, text_input, lbl, return_sal=False, modality='both'):
    nb_text_tokens = text_input.input_ids.shape[1]
    p = int(math.ceil(np.sqrt(nb_text_tokens)))
    patch_size = 384 // p
    image_token_ids = torch.tensor(range(1, p**2 + 1)).unsqueeze(0)
    X = torch.cat((text_input.input_ids, image_token_ids), 1).unsqueeze(1)

    def custom_masker(mask, x):
        masked_X = x.clone()
        mask = torch.tensor(mask).unsqueeze(0)
        masked_X[~mask] = 50264
        masked_X[0, 0] = 0  # CLS
        masked_X[0, nb_text_tokens - 1] = 2  # SEP
        return masked_X

    def get_model_prediction(x): 
        with torch.no_grad():
            input_ids = torch.tensor(x[:, :text_input.input_ids.shape[1]])
            masked_image_token_ids = torch.tensor(x[:, text_input.input_ids.shape[1]:])
            if torch.cuda.is_available():
                input_ids = input_ids.cuda()
                masked_image_token_ids = masked_image_token_ids.cuda()
            result = np.zeros(input_ids.shape[0])
            row_cols = 384 // patch_size
            for i in range(input_ids.shape[0]):
                masked_text_inputs = text_input.copy()
                masked_text_inputs['input_ids'] = input_ids[i].unsqueeze(0)
                masked_image = copy.deepcopy(image_input)
                
                for k in range(masked_image_token_ids[i].shape[0]):
                    if masked_image_token_ids[i][k] == 50264:
                        m = k // row_cols
                        n = k % row_cols
                        masked_image[:, :, m * patch_size:(m + 1) * patch_size, n * patch_size:(n + 1) * patch_size] = 0
                outputs = model(
                    masked_text_inputs['input_ids'].to(DEVICE),
                    masked_text_inputs['attention_mask'].to(DEVICE),
                    masked_image.to(DEVICE)
                )
                result[i] = torch.nn.Softmax(dim=1)(outputs).cpu().detach()[:, 1]
        return result

    # Run SHAP with random seed
    explainer = shap.Explainer(get_model_prediction, custom_masker, silent=True)#, seed=RANDOM_STATE)
    shap_values = explainer(X, max_evals=577, batch_size=20)  # optional: set batch_size for determinism

    img_patch_vals = shap_values.values[0][0][text_input.input_ids.shape[1]:]
    txt_patch_vals = shap_values.values[0][0][:text_input.input_ids.shape[1]]

    # Set random state in sorting if using randomness (though sorted() is deterministic)
    sorted_scores_txt_shap = sorted([(i, val) for i, val in enumerate(txt_patch_vals)],
                                    key=lambda x: x[1], reverse=True)
    sorted_scores_img_shap = sorted([(i, val) for i, val in enumerate(img_patch_vals)],
                                    key=lambda x: x[1], reverse=True)
    
    if return_sal == True:
        return sorted_scores_img_shap, sorted_scores_txt_shap

    # individually
    if modality == 'indiv':
        scores_img_del_shap, correct1, out1 = insertion_deletion(image_input, text_input, lbl, sorted_scores_img_shap, 'del', 'image', return_output_changes=True, masking_type='zero')
        scores_img_ins_shap, _, out2 = insertion_deletion(image_input, text_input, lbl, sorted_scores_img_shap, 'ins', 'image', return_output_changes=True, masking_type='blur')
        scores_txt_del_shap, _, out3 = insertion_deletion(image_input, text_input, lbl, sorted_scores_txt_shap, 'del', 'text', return_output_changes=True)
        scores_txt_ins_shap, _, out4 = insertion_deletion(image_input, text_input, lbl, sorted_scores_txt_shap, 'ins', 'text', return_output_changes=True)
        auc_img_del_shap = auc(np.linspace(0, 1, len(scores_img_del_shap)), scores_img_del_shap)
        auc_img_ins_shap = auc(np.linspace(0, 1, len(scores_img_ins_shap)), scores_img_ins_shap)
        auc_txt_del_shap = auc(np.linspace(0, 1, len(scores_txt_del_shap)), scores_txt_del_shap)
        auc_txt_ins_shap = auc(np.linspace(0, 1, len(scores_txt_ins_shap)), scores_txt_ins_shap)
        img_feature_importances = [score for _, score in sorted_scores_img_shap]
        txt_feature_importances = [score for _, score in sorted_scores_txt_shap]
        faith_img_del_shap = spearmanr(img_feature_importances, out1).correlation
        faith_img_ins_shap = spearmanr(img_feature_importances, out2).correlation
        faith_txt_del_shap = spearmanr(txt_feature_importances, out3).correlation
        faith_txt_ins_shap = spearmanr(txt_feature_importances, out4).correlation
        return scores_img_del_shap, scores_txt_del_shap, scores_img_ins_shap, scores_txt_ins_shap, auc_img_del_shap, auc_txt_del_shap, auc_img_ins_shap, auc_txt_ins_shap, faith_img_del_shap, faith_txt_del_shap, faith_img_ins_shap, faith_txt_ins_shap, correct1

    # together
    if modality == 'both':
        scores_del, correct1, out1 = insertion_deletion_both(image_input, text_input, lbl, sorted_scores_img_shap, sorted_scores_txt_shap, 'del', return_output_changes=True, masking_type='zero')
        scores_ins, _, out2 = insertion_deletion_both(image_input, text_input, lbl, sorted_scores_img_shap, sorted_scores_txt_shap, 'ins', return_output_changes=True, masking_type='blur')
        auc_del = auc(np.linspace(0, 1, len(scores_del)), scores_del)
        auc_ins = auc(np.linspace(0, 1, len(scores_ins)), scores_ins)
        img_feature_importances = [score for _, score in sorted_scores_img_shap]
        txt_feature_importances = [score for _, score in sorted_scores_txt_shap]
        spearman_del_img = spearmanr(img_feature_importances, out1).correlation # correlation between the (img feature importance scores) and (the changes in prob when the next best token AND patch is masked)
        spearman_ins_img = spearmanr(img_feature_importances, out2).correlation # correlation between the (img feature importance scores) and (the changes in prob when the next best token AND patch is unmasked)
        spearman_del_txt = spearmanr(txt_feature_importances, out1).correlation # correlation between the (txt feature importance scores) and (the changes in prob when the next best token AND patch is masked)
        spearman_ins_txt = spearmanr(txt_feature_importances, out2).correlation # correlation between the (txt feature importance scores) and (the changes in prob when the next best token AND patch is unmasked)
        return scores_del, scores_ins, auc_del, auc_ins, spearman_del_img, spearman_ins_img, spearman_del_txt, spearman_ins_txt, correct1

def compute_rise_auc_scores(image_input, text_input, lbl, num_token_masks=144, seq_length=144, token_mask_prob=0.5, num_image_masks=144, grid_size=12, image_mask_prob=0.5, return_sal=False, modality='both'):

    def generate_token_masks(num_masks, seq_length, probability_ofmask):
        masks = np.random.rand(num_masks, seq_length) < probability_ofmask
        return masks.astype(np.float32)
    
    def generate_binary_masks(grid_size, probability_ofmask, num_masks):
        shape = (384, 384)
        cell_size = np.ceil(np.array(shape) / grid_size).astype(int)
        up_size = ((grid_size + 1) * cell_size).astype(int)
        grid = np.random.rand(num_masks, grid_size, grid_size) < probability_ofmask
        grid = grid.astype('float32')
        masks = np.empty((num_masks, *shape))
        for i in range(num_masks):
            x = np.random.randint(0, cell_size[0])
            y = np.random.randint(0, cell_size[1])
            upsampled = resize(grid[i], up_size, order=1, mode='reflect', anti_aliasing=False)
            masks[i, :, :] = upsampled[x:x + shape[0], y:y + shape[1]]
        return masks.reshape(-1, *shape, 1)

    def explain_tokens(masks):
        probs = []
        for i in range(masks.shape[0]):
            masked_tokens = copy.deepcopy(text_input)
            curmask = np.expand_dims(masks[i], axis=0)
            masked_tokens['input_ids'][curmask == 1] = 50264
            outputs = query_model(image_input, masked_tokens)
            softmax_output = torch.softmax(outputs, dim=1)
            val = softmax_output[0][baseline_class].cpu().numpy()
            probs.append(np.array([val]))
        preds = np.concatenate(probs)
        saliency = preds.T.dot(masks) / masks.shape[0]
        return saliency

    def explain_image(masks):
        probs = []
        for i in range(masks.shape[0]):
            currentmask = masks[i]
            org_img = copy.deepcopy(image_input).detach().cpu().numpy().squeeze()
            currentmask = np.transpose(currentmask, (2, 0, 1))
            masked_img = currentmask * org_img
            masked_img_tensor = torch.tensor(masked_img).unsqueeze(0)
            outputs = query_model(masked_img_tensor, text_input)
            softmax_output = torch.softmax(outputs, dim=1)
            val = softmax_output[0][baseline_class].cpu().numpy()
            probs.append(np.array([val]))
        preds = np.concatenate(probs)
        sal = preds.T.dot(masks.reshape(masks.shape[0], -1)).reshape(-1, 384, 384)
        sal = sal / masks.shape[0] / image_mask_prob
        return sal
    
    outputs = query_model(image_input, text_input)
    softmax_output = torch.softmax(outputs, dim=1)
    _, cat = torch.max(softmax_output, -1)
    baseline_class = int(cat.cpu().numpy()[0])

    # === Text Saliency ===
    token_masks = generate_token_masks(num_token_masks, seq_length, token_mask_prob)
    saliency_text = explain_tokens(token_masks)
    sorted_scores_txt_rise = sorted([(i, val) for i, val in enumerate(saliency_text)], key=lambda x: x[1], reverse=True)

    # === Image Saliency ===
    image_masks = generate_binary_masks(grid_size, image_mask_prob, num_image_masks)
    saliency_img = explain_image(image_masks)
    normalized_img = (saliency_img - saliency_img.min()) / (saliency_img.max() - saliency_img.min())

    img_saliency_discrete = np.zeros((grid_size, grid_size))
    scores_img = []
    cell_size = 384 // grid_size
    idx = 0
    for row in range(grid_size):
        for col in range(grid_size):
            r0, r1 = row * cell_size, (row + 1) * cell_size
            c0, c1 = col * cell_size, (col + 1) * cell_size
            avg_val = np.mean(normalized_img[:, r0:r1, c0:c1])
            img_saliency_discrete[row, col] = avg_val
            scores_img.append((idx, avg_val))
            idx += 1
    sorted_scores_img_rise = sorted(scores_img, key=lambda x: x[1], reverse=True)
 
    if return_sal == True:
        return sorted_scores_img_rise, sorted_scores_txt_rise
    
    # individually
    if modality == 'indiv':
        scores_img_del_rise, correct1, out1 = insertion_deletion(image_input, text_input, lbl, sorted_scores_img_rise, 'del', 'image', return_output_changes=True, masking_type='zero')
        scores_img_ins_rise, _, out2 = insertion_deletion(image_input, text_input, lbl, sorted_scores_img_rise, 'ins', 'image', return_output_changes=True, masking_type='blur')
        scores_txt_del_rise, _, out3 = insertion_deletion(image_input, text_input, lbl, sorted_scores_txt_rise, 'del', 'text', return_output_changes=True)
        scores_txt_ins_rise, _, out4 = insertion_deletion(image_input, text_input, lbl, sorted_scores_txt_rise, 'ins', 'text', return_output_changes=True)
        auc_img_del_rise = auc(np.linspace(0, 1, len(scores_img_del_rise)), scores_img_del_rise)
        auc_img_ins_rise = auc(np.linspace(0, 1, len(scores_img_ins_rise)), scores_img_ins_rise)
        auc_txt_del_rise = auc(np.linspace(0, 1, len(scores_txt_del_rise)), scores_txt_del_rise)
        auc_txt_ins_rise = auc(np.linspace(0, 1, len(scores_txt_ins_rise)), scores_txt_ins_rise)
        img_feature_importances = [score for _, score in sorted_scores_img_rise]
        txt_feature_importances = [score for _, score in sorted_scores_txt_rise]
        faith_img_del_rise = spearmanr(img_feature_importances, out1).correlation
        faith_img_ins_rise = spearmanr(img_feature_importances, out2).correlation
        faith_txt_del_rise = spearmanr(txt_feature_importances, out3).correlation
        faith_txt_ins_rise = spearmanr(txt_feature_importances, out4).correlation
        return scores_img_del_rise, scores_txt_del_rise, scores_img_ins_rise, scores_txt_ins_rise, auc_img_del_rise, auc_txt_del_rise, auc_img_ins_rise, auc_txt_ins_rise, faith_img_del_rise, faith_txt_del_rise, faith_img_ins_rise, faith_txt_ins_rise, correct1

    # together
    if modality == 'both':
        scores_del, correct1, out1 = insertion_deletion_both(image_input, text_input, lbl, sorted_scores_img_rise, sorted_scores_txt_rise, 'del', return_output_changes=True, masking_type='zero')
        scores_ins, _, out2 = insertion_deletion_both(image_input, text_input, lbl, sorted_scores_img_rise, sorted_scores_txt_rise, 'ins', return_output_changes=True, masking_type='blur')
        auc_del = auc(np.linspace(0, 1, len(scores_del)), scores_del)
        auc_ins = auc(np.linspace(0, 1, len(scores_ins)), scores_ins)
        img_feature_importances = [score for _, score in sorted_scores_img_rise]
        txt_feature_importances = [score for _, score in sorted_scores_txt_rise]
        spearman_del_img = spearmanr(img_feature_importances, out1).correlation
        spearman_ins_img = spearmanr(img_feature_importances, out2).correlation
        spearman_del_txt = spearmanr(txt_feature_importances, out1).correlation
        spearman_ins_txt = spearmanr(txt_feature_importances, out2).correlation
        return scores_del, scores_ins, auc_del, auc_ins, spearman_del_img, spearman_ins_img, spearman_del_txt, spearman_ins_txt, correct1

def compute_lime_auc_scores(image_input, text_input, lbl, return_sal=False, num_samples=144, modality='both'):

    # === TEXT PERTURBATION (binary mask representation) ===
    def perturb_text(text_input, num_samples):
        input_ids = text_input['input_ids'].cpu().numpy()  # shape (1, seq_len)
        seq_length = input_ids.shape[1]
        masks = np.random.randint(0, 2, size=(num_samples, seq_length))  # binary masks
        perturbed_inputs = []
        for mask in masks:
            perturbed_input = input_ids.copy()  # shape (1, seq_len)
            perturbed_input[0][mask == 0] = 50264  # apply mask token
            perturbed_inputs.append(perturbed_input)
        return np.array(perturbed_inputs), masks

    # === IMAGE PERTURBATION (binary mask representation) ===
    def perturb_image(image_input, num_samples):
        image_np = image_input.cpu().numpy()
        num_patches = 144  # 12x12 grid
        masks = np.random.randint(0, 2, size=(num_samples, num_patches))
        perturbed_images = []
        mask_views = []
        for i, mask in enumerate(masks):
            perturbed_image = image_np.copy()
            mask_view = np.ones_like(image_np)
            for j in range(num_patches):
                if mask[j] == 0:
                    row, col = divmod(j, 12)
                    perturbed_image[:, :, row*32:(row+1)*32, col*32:(col+1)*32] = 0
                    mask_view[:, :, row*32:(row+1)*32, col*32:(col+1)*32] = 0
            perturbed_images.append(perturbed_image)
            mask_views.append(mask_view)
        return np.array(perturbed_images), masks

    perturbed_texts, text_masks = perturb_text(text_input, num_samples)
    perturbed_images, image_masks = perturb_image(image_input, num_samples)

    # === Get predictions ===
    def get_model_predictions(perturbed_texts, perturbed_images):
        model.eval()
        preds = []
        with torch.no_grad():
            for i in range(num_samples):
                input_ids_tensor = torch.tensor(perturbed_texts[i]).to(DEVICE)
                attention_mask_tensor = (input_ids_tensor != 50264).long().to(DEVICE)
                image_tensor = torch.tensor(perturbed_images[i]).to(DEVICE)
                outputs = model(input_ids_tensor, attention_mask_tensor, image_tensor)
                softmax_output = torch.softmax(outputs, dim=1)
                preds.append(softmax_output[0].cpu().numpy())
        return np.array(preds)

    preds = get_model_predictions(perturbed_texts, perturbed_images)
    original_output = query_model(image_input, text_input)
    softmax_output = torch.softmax(original_output, dim=1).cpu().numpy()
    baseline_class = np.argmax(softmax_output)
    preds = preds[:, baseline_class]  # predicted probability of the target class

    # === Build binary mask feature space ===
    combined_masks = np.hstack((text_masks, image_masks))

    # === Compute proximity weights (locality kernel) ===
    distances = cosine_distances(combined_masks, np.ones((1, combined_masks.shape[1])))
    kernel_width = 0.25 * np.sqrt(combined_masks.shape[1])  # common heuristic
    weights = np.sqrt(np.exp(-(distances**2) / (kernel_width**2))).flatten()

    # === Fit surrogate model ===
    reg = Ridge(alpha=1.0)
    reg.fit(combined_masks, preds, sample_weight=weights)
    coefficients = reg.coef_.flatten()

    # Split coefficients
    text_coefficients = coefficients[:text_input['input_ids'].shape[1]]
    image_coefficients = coefficients[text_input['input_ids'].shape[1]:]

    sorted_scores_txt_lime = sorted([(i, float(score)) for i, score in enumerate(text_coefficients)], key=lambda x: -x[1])
    sorted_scores_img_lime = sorted([(i, float(score)) for i, score in enumerate(image_coefficients)], key=lambda x: -x[1])

    if return_sal:
        return sorted_scores_img_lime, sorted_scores_txt_lime

    # Compute insertion/deletion AUCs
    if modality == 'both':
        scores_del, correct1, out1 = insertion_deletion_both(image_input, text_input, lbl, sorted_scores_img_lime, sorted_scores_txt_lime, 'del', return_output_changes=True, masking_type='zero')
        scores_ins, _, out2 = insertion_deletion_both(image_input, text_input, lbl, sorted_scores_img_lime, sorted_scores_txt_lime, 'ins', return_output_changes=True, masking_type='blur')
        auc_del = auc(np.linspace(0, 1, len(scores_del)), scores_del)
        auc_ins = auc(np.linspace(0, 1, len(scores_ins)), scores_ins)
        return sorted_scores_img_lime, sorted_scores_txt_lime, scores_del, scores_ins, auc_del, auc_ins, correct1
    
    if modality == 'indiv':
        scores_img_del_lime, correct1, out1 = insertion_deletion(image_input, text_input, lbl, sorted_scores_img_lime, 'del', 'image', return_output_changes=True, masking_type='zero')
        scores_img_ins_lime, _, out2 = insertion_deletion(image_input, text_input, lbl, sorted_scores_img_lime, 'ins', 'image', return_output_changes=True, masking_type='blur')
        scores_txt_del_lime, _, out3 = insertion_deletion(image_input, text_input, lbl, sorted_scores_txt_lime, 'del', 'text', return_output_changes=True)
        scores_txt_ins_lime, _, out4 = insertion_deletion(image_input, text_input, lbl, sorted_scores_txt_lime, 'ins', 'text', return_output_changes=True)
        auc_img_del_lime = auc(np.linspace(0, 1, len(scores_img_del_lime)), scores_img_del_lime)
        auc_img_ins_lime = auc(np.linspace(0, 1, len(scores_img_ins_lime)), scores_img_ins_lime)
        auc_txt_del_lime = auc(np.linspace(0, 1, len(scores_txt_del_lime)), scores_txt_del_lime)
        auc_txt_ins_lime = auc(np.linspace(0, 1, len(scores_txt_ins_lime)), scores_txt_ins_lime)
        return scores_img_del_lime, scores_txt_del_lime, scores_img_ins_lime, scores_txt_ins_lime, auc_img_del_lime, auc_txt_del_lime, auc_img_ins_lime, auc_txt_ins_lime, correct1

# attention based
def get_vanilla_auc_scores(image_input, text_input, lbl, return_sal=False, modality='both'):
    input_ids = text_input['input_ids'].to(DEVICE)
    attention_mask = text_input['attention_mask'].to(DEVICE)
    image_input = image_input.to(DEVICE)
    image_input.requires_grad_(True)

    # Forward pass with attentions
    output, text_attentions, image_attentions = model(input_ids, attention_mask, image_input, output_attentions=True)

    # --- TEXT ATTENTION PROCESSING ---
    last_layer_attention_text = text_attentions[-1]  # (batch, heads, seq_len, seq_len)
    last_layer_attention_text_avg = last_layer_attention_text.mean(dim=1)  # (batch, seq_len, seq_len)
    attention_map_text = last_layer_attention_text_avg[0].detach().cpu().numpy()

    seq_len = attention_map_text.shape[0]
    scores_txt = []
    for i in range(seq_len):
        attention_scores = [attention_map_text[j, i] for j in range(seq_len)]
        attention_avg = sum(attention_scores) / seq_len
        scores_txt.append((i, attention_avg))
    sorted_scores_txt_van = sorted(scores_txt, key=lambda x: x[1], reverse=True)

    # --- IMAGE ATTENTION PROCESSING ---
    last_layer_attention_img = image_attentions[-1]  # (batch, heads, num_patches, num_patches)
    last_attention_block_img_avg = last_layer_attention_img.mean(dim=1)  # (batch, num_patches, num_patches)
    attention_map_img = last_attention_block_img_avg[0].detach().cpu().numpy()

    seq_len = attention_map_img.shape[0]
    scores_img = []
    for i in range(seq_len):
        attention_scores = [attention_map_img[j, i] for j in range(seq_len)]
        attention_avg = sum(attention_scores) / seq_len
        scores_img.append((i, attention_avg))
    sorted_scores_img = sorted(scores_img, key=lambda x: x[1], reverse=True)

    # Construct saliency map
    img_saliency = copy.deepcopy(image_input).squeeze().detach().cpu().numpy()
    for i, (_, score) in enumerate(sorted_scores_img):
        patch_idx = sorted_scores_img[i][0]
        row = patch_idx // 24
        col = patch_idx % 24
        start_row = row * 16
        end_row = (row + 1) * 16
        start_col = col * 16
        end_col = (col + 1) * 16
        img_saliency[:, start_row:end_row, start_col:end_col] = score

    # Downsample to 12x12
    img_saliency_downsampled = np.zeros((12, 12))
    downsampled_scores = []
    ctr = 0
    for row in range(12):
        for col in range(12):
            start_row = row * 32
            end_row = (row + 1) * 32
            start_col = col * 32
            end_col = (col + 1) * 32
            avg_val = np.mean(np.unique(img_saliency[:, start_row:end_row, start_col:end_col]))
            img_saliency_downsampled[row, col] = avg_val
            downsampled_scores.append((ctr, avg_val))
            ctr += 1
    sorted_scores_img_van = sorted(downsampled_scores, key=lambda x: x[1], reverse=True)

    if return_sal == True:
        return sorted_scores_img_van, sorted_scores_txt_van

    # individually
    if modality == 'indiv':
        scores_img_del_van, correct1, out1 = insertion_deletion(image_input, text_input, lbl, sorted_scores_img_van, 'del', 'image', return_output_changes=True, masking_type='zero')
        scores_img_ins_van, _, out2 = insertion_deletion(image_input, text_input, lbl, sorted_scores_img_van, 'ins', 'image', return_output_changes=True, masking_type='blur')
        scores_txt_del_van, _, out3 = insertion_deletion(image_input, text_input, lbl, sorted_scores_txt_van, 'del', 'text', return_output_changes=True)
        scores_txt_ins_van, _, out4 = insertion_deletion(image_input, text_input, lbl, sorted_scores_txt_van, 'ins', 'text', return_output_changes=True)
        auc_img_del_van = auc(np.linspace(0, 1, len(scores_img_del_van)), scores_img_del_van)
        auc_img_ins_van = auc(np.linspace(0, 1, len(scores_img_ins_van)), scores_img_ins_van)
        auc_txt_del_van = auc(np.linspace(0, 1, len(scores_txt_del_van)), scores_txt_del_van)
        auc_txt_ins_van = auc(np.linspace(0, 1, len(scores_txt_ins_van)), scores_txt_ins_van)
        img_feature_importances = [score for _, score in sorted_scores_img_van]
        txt_feature_importances = [score for _, score in sorted_scores_txt_van]
        faith_img_del_van = spearmanr(img_feature_importances, out1).correlation
        faith_img_ins_van = spearmanr(img_feature_importances, out2).correlation
        faith_txt_del_van = spearmanr(txt_feature_importances, out3).correlation
        faith_txt_ins_van = spearmanr(txt_feature_importances, out4).correlation
        return scores_img_del_van, scores_txt_del_van, scores_img_ins_van, scores_txt_ins_van, auc_img_del_van, auc_txt_del_van, auc_img_ins_van, auc_txt_ins_van, faith_img_del_van, faith_txt_del_van, faith_img_ins_van, faith_txt_ins_van, correct1

    # together
    if modality == 'both':
        scores_del, correct1, out1 = insertion_deletion_both(image_input, text_input, lbl, sorted_scores_img_van, sorted_scores_txt_van, 'del', return_output_changes=True, masking_type='zero')
        scores_ins, _, out2 = insertion_deletion_both(image_input, text_input, lbl, sorted_scores_img_van, sorted_scores_txt_van, 'ins', return_output_changes=True, masking_type='blur')
        auc_del = auc(np.linspace(0, 1, len(scores_del)), scores_del)
        auc_ins = auc(np.linspace(0, 1, len(scores_ins)), scores_ins)
        img_feature_importances = [score for _, score in sorted_scores_img_van]
        txt_feature_importances = [score for _, score in sorted_scores_txt_van]
        spearman_del_img = spearmanr(img_feature_importances, out1).correlation # correlation between the (img feature importance scores) and (the changes in prob when the next best token AND patch is masked)
        spearman_ins_img = spearmanr(img_feature_importances, out2).correlation # correlation between the (img feature importance scores) and (the changes in prob when the next best token AND patch is unmasked)
        spearman_del_txt = spearmanr(txt_feature_importances, out1).correlation # correlation between the (txt feature importance scores) and (the changes in prob when the next best token AND patch is masked)
        spearman_ins_txt = spearmanr(txt_feature_importances, out2).correlation # correlation between the (txt feature importance scores) and (the changes in prob when the next best token AND patch is unmasked)
        return scores_del, scores_ins, auc_del, auc_ins, spearman_del_img, spearman_ins_img, spearman_del_txt, spearman_ins_txt, correct1

def compute_rollout_auc_scores(image_input, text_input, lbl, return_sal=False, modality='both'):
    model.eval()
    # Manually add one extra padding token to keep the length correct
    text_input['input_ids'] = torch.cat([text_input['input_ids'], torch.tensor([[1]], dtype=torch.long)], dim=1)
    text_input['attention_mask'] = torch.cat([text_input['attention_mask'], torch.tensor([[0]], dtype=torch.long)], dim=1)
    input_ids = text_input['input_ids'].to(DEVICE)
    attention_mask = text_input['attention_mask'].to(DEVICE)
    image_input = image_input.to(DEVICE)
    image_input.requires_grad_(True)

    def compute_rollout_attention(all_layer_matrices, start_layer=0):
        batch_size, num_heads, tokens, _ = all_layer_matrices[0].shape
        eye = torch.eye(tokens).expand(batch_size, tokens, tokens).to(all_layer_matrices[0].device)
        joint = [attn.mean(dim=1) + eye for attn in all_layer_matrices]  # Average heads + identity
        joint = [j / j.sum(dim=-1, keepdim=True) for j in joint]         # Normalize

        rollout = joint[start_layer]
        for mat in joint[start_layer + 1:]:
            rollout = mat.bmm(rollout)
        return rollout  # (B, tokens, tokens)

    # Forward pass
    with torch.no_grad():
        logits, text_attns, image_attns = model(input_ids, attention_mask, image_input, output_attentions=True)

    # -----------------------------
    # Text Token Attention Rollout
    # -----------------------------
    text_rollout = compute_rollout_attention(text_attns)
    text_importance = text_rollout[0, 0, 1:]  # Skip CLS token
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    #token_importance = [(tok, round(score.item(), 4)) for tok, score in zip(tokens[1:], text_importance)]
    sorted_scores_txt_roll = sorted([(i + 1, np.float32(score.item())) for i, score in enumerate(text_importance)],key=lambda x: -x[1])

    # -----------------------------
    # Image Patch Attention Rollout
    # -----------------------------
    image_rollout = compute_rollout_attention(image_attns)
    img_mask = image_rollout[0, 0, 1:]  # Skip CLS
    img_mask = img_mask.reshape(24, 24).cpu().numpy()
    img_mask = cv2.resize(img_mask, (384, 384))
    img_mask = (img_mask - img_mask.min()) / (img_mask.max() - img_mask.min())

    img_saliency_downsampled = np.zeros((12,12))
    downsampled_scores = []
    ctr = 0
    for row in range(12):
        for col in range(12):
            start_row = row * 32
            end_row = (row + 1) * 32
            start_col = col * 32
            end_col = (col + 1) * 32
            avg_saliency_val = np.mean(img_mask[start_row:end_row, start_col:end_col])
            img_saliency_downsampled[row, col] = avg_saliency_val
            downsampled_scores.append((ctr, avg_saliency_val))
            ctr += 1
    sorted_scores_img_roll = sorted(downsampled_scores, key=lambda x: x[1], reverse=True)

    if return_sal == True:
        return sorted_scores_img_roll, sorted_scores_txt_roll

    # individually
    if modality == 'indiv':
        scores_img_del_roll, correct1, out1 = insertion_deletion(image_input, text_input, lbl, sorted_scores_img_roll, 'del', 'image', return_output_changes=True, masking_type='zero')
        scores_img_ins_roll, _, out2 = insertion_deletion(image_input, text_input, lbl, sorted_scores_img_roll, 'ins', 'image', return_output_changes=True, masking_type='blur')
        scores_txt_del_roll, _, out3 = insertion_deletion(image_input, text_input, lbl, sorted_scores_txt_roll, 'del', 'text', return_output_changes=True)
        scores_txt_ins_roll, _, out4 = insertion_deletion(image_input, text_input, lbl, sorted_scores_txt_roll, 'ins', 'text', return_output_changes=True)
        auc_img_del_roll = auc(np.linspace(0, 1, len(scores_img_del_roll)), scores_img_del_roll)
        auc_img_ins_roll = auc(np.linspace(0, 1, len(scores_img_ins_roll)), scores_img_ins_roll)
        auc_txt_del_roll = auc(np.linspace(0, 1, len(scores_txt_del_roll)), scores_txt_del_roll)
        auc_txt_ins_roll = auc(np.linspace(0, 1, len(scores_txt_ins_roll)), scores_txt_ins_roll)
        img_feature_importances = [score for _, score in sorted_scores_img_roll]
        txt_feature_importances = [score for _, score in sorted_scores_txt_roll]
        faith_img_del_roll = spearmanr(img_feature_importances, out1).correlation
        faith_img_ins_roll = spearmanr(img_feature_importances, out2).correlation
        faith_txt_del_roll = spearmanr(txt_feature_importances, out3).correlation
        faith_txt_ins_roll = spearmanr(txt_feature_importances, out4).correlation
        return scores_img_del_roll, scores_txt_del_roll, scores_img_ins_roll, scores_txt_ins_roll, auc_img_del_roll, auc_txt_del_roll, auc_img_ins_roll, auc_txt_ins_roll, faith_img_del_roll, faith_txt_del_roll, faith_img_ins_roll, faith_txt_ins_roll, correct1

    # together
    if modality == 'both':
        scores_del, correct1, out1 = insertion_deletion_both(image_input, text_input, lbl, sorted_scores_img_roll, sorted_scores_txt_roll, 'del', return_output_changes=True, masking_type='zero')
        scores_ins, _, out2 = insertion_deletion_both(image_input, text_input, lbl, sorted_scores_img_roll, sorted_scores_txt_roll, 'ins', return_output_changes=True, masking_type='blur')
        auc_del = auc(np.linspace(0, 1, len(scores_del)), scores_del)
        auc_ins = auc(np.linspace(0, 1, len(scores_ins)), scores_ins)
        img_feature_importances = [score for _, score in sorted_scores_img_roll]
        txt_feature_importances = [score for _, score in sorted_scores_txt_roll]
        spearman_del_img = spearmanr(img_feature_importances, out1).correlation # correlation between the (img feature importance scores) and (the changes in prob when the next best token AND patch is masked)
        spearman_ins_img = spearmanr(img_feature_importances, out2).correlation # correlation between the (img feature importance scores) and (the changes in prob when the next best token AND patch is unmasked)
        spearman_del_txt = spearmanr(txt_feature_importances, out1).correlation # correlation between the (txt feature importance scores) and (the changes in prob when the next best token AND patch is masked)
        spearman_ins_txt = spearmanr(txt_feature_importances, out2).correlation # correlation between the (txt feature importance scores) and (the changes in prob when the next best token AND patch is unmasked)
        return scores_del, scores_ins, auc_del, auc_ins, spearman_del_img, spearman_ins_img, spearman_del_txt, spearman_ins_txt, correct1

# random baseline
def compute_random_baseline(image_input, text_input, lbl, idx, return_sal=False, modality='both'):
    # random.seed(idx)
    # np.random.seed(idx)
    np_rnd = np.random.default_rng(idx)  # local NumPy RNG

    sorted_scores_img_rand = sorted([(i, np.float64(np_rnd.random())) for i in list(range(144))], key=lambda x: x[1], reverse=True)
    sorted_scores_txt_rand = sorted([(i, np.float64(np_rnd.random())) for i in list(range(144))], key=lambda x: x[1], reverse=True)

    if return_sal == True:
        return sorted_scores_img_rand, sorted_scores_txt_rand

    # individually
    if modality == 'indiv':
        scores_img_del_rand, correct1, out1 = insertion_deletion(image_input, text_input, lbl, sorted_scores_img_rand, 'del', 'image', return_output_changes=True, masking_type='zero')
        scores_img_ins_rand, _, out2 = insertion_deletion(image_input, text_input, lbl, sorted_scores_img_rand, 'ins', 'image', return_output_changes=True, masking_type='blur')
        scores_txt_del_rand, _, out3 = insertion_deletion(image_input, text_input, lbl, sorted_scores_txt_rand, 'del', 'text', return_output_changes=True)
        scores_txt_ins_rand, _, out4 = insertion_deletion(image_input, text_input, lbl, sorted_scores_txt_rand, 'ins', 'text', return_output_changes=True)
        auc_img_del_rand = auc(np.linspace(0, 1, len(scores_img_del_rand)), scores_img_del_rand)
        auc_img_ins_rand = auc(np.linspace(0, 1, len(scores_img_ins_rand)), scores_img_ins_rand)
        auc_txt_del_rand = auc(np.linspace(0, 1, len(scores_txt_del_rand)), scores_txt_del_rand)
        auc_txt_ins_rand = auc(np.linspace(0, 1, len(scores_txt_ins_rand)), scores_txt_ins_rand)
        img_feature_importances = [score for _, score in sorted_scores_img_rand]
        txt_feature_importances = [score for _, score in sorted_scores_txt_rand]
        faith_img_del_rand = spearmanr(img_feature_importances, out1).correlation
        faith_img_ins_rand = spearmanr(img_feature_importances, out2).correlation
        faith_txt_del_rand = spearmanr(txt_feature_importances, out3).correlation
        faith_txt_ins_rand = spearmanr(txt_feature_importances, out4).correlation
        return scores_img_del_rand, scores_txt_del_rand, scores_img_ins_rand, scores_txt_ins_rand, auc_img_del_rand, auc_txt_del_rand, auc_img_ins_rand, auc_txt_ins_rand, faith_img_del_rand, faith_txt_del_rand, faith_img_ins_rand, faith_txt_ins_rand, correct1

    # together
    if modality == 'both':
        scores_del, correct1, out1 = insertion_deletion_both(image_input, text_input, lbl, sorted_scores_img_rand, sorted_scores_txt_rand, 'del', return_output_changes=True, masking_type='zero')
        scores_ins, _, out2 = insertion_deletion_both(image_input, text_input, lbl, sorted_scores_img_rand, sorted_scores_txt_rand, 'ins', return_output_changes=True, masking_type='blur')
        auc_del = auc(np.linspace(0, 1, len(scores_del)), scores_del)
        auc_ins = auc(np.linspace(0, 1, len(scores_ins)), scores_ins)
        img_feature_importances = [score for _, score in sorted_scores_img_rand]
        txt_feature_importances = [score for _, score in sorted_scores_txt_rand]
        spearman_del_img = spearmanr(img_feature_importances, out1).correlation # correlation between the (img feature importance scores) and (the changes in prob when the next best token AND patch is masked)
        spearman_ins_img = spearmanr(img_feature_importances, out2).correlation # correlation between the (img feature importance scores) and (the changes in prob when the next best token AND patch is unmasked)
        spearman_del_txt = spearmanr(txt_feature_importances, out1).correlation # correlation between the (txt feature importance scores) and (the changes in prob when the next best token AND patch is masked)
        spearman_ins_txt = spearmanr(txt_feature_importances, out2).correlation # correlation between the (txt feature importance scores) and (the changes in prob when the next best token AND patch is unmasked)
        return scores_del, scores_ins, auc_del, auc_ins, spearman_del_img, spearman_ins_img, spearman_del_txt, spearman_ins_txt, correct1

In [ ]:
balanced_df = test_df.groupby('voted_label').apply(lambda x: x.sample(n=8, random_state=0)).droplevel(0)
print(balanced_df['voted_label'].value_counts())

In [ ]:
# all methods run
# For 26 samples:
# together:     51min-40.9s - 3:30:20
# individually: 69min-22.3s - 4:31:28
ctr = 0
for original_idx, row in tqdm(balanced_df.iterrows(), total=len(balanced_df), desc="Processing samples"):
    # if ctr == 1:
    #     break
    padding_len = 144
    text = row['text']
    test_img_path = '../' + row['image']
    lbl = row['voted_label']

    image_pil = Image.open(test_img_path).convert('RGB')
    image_input = image_processor(image_pil, return_tensors="pt")['pixel_values']
    text_input = tokenizer(text, return_tensors="pt", padding='max_length', max_length=padding_len)

    # for image and text separately
    scores_img_del_shap, scores_txt_del_shap, scores_img_ins_shap, scores_txt_ins_shap, auc_img_del_shap, auc_txt_del_shap, auc_img_ins_shap, auc_txt_ins_shap, faith_img_del_shap, faith_txt_del_shap, faith_img_ins_shap, faith_txt_ins_shap, correct_baseline_pred1 = compute_shap_auc_scores(image_input, text_input, lbl)
    scores_img_del_rise, scores_txt_del_rise, scores_img_ins_rise, scores_txt_ins_rise, auc_img_del_rise, auc_txt_del_rise, auc_img_ins_rise, auc_txt_ins_rise, faith_img_del_rise, faith_txt_del_rise, faith_img_ins_rise, faith_txt_ins_rise, correct_baseline_pred2 = compute_rise_auc_scores(image_input, text_input, lbl)
    scores_img_del_van, scores_txt_del_van, scores_img_ins_van, scores_txt_ins_van, auc_img_del_van, auc_txt_del_van, auc_img_ins_van, auc_txt_ins_van, faith_img_del_van, faith_txt_del_van, faith_img_ins_van, faith_txt_ins_van, correct_baseline_pred3 = get_vanilla_auc_scores(image_input, text_input, lbl)
    scores_img_del_roll, scores_txt_del_roll, scores_img_ins_roll, scores_txt_ins_roll, auc_img_del_roll, auc_txt_del_roll, auc_img_ins_roll, auc_txt_ins_roll, faith_img_del_roll, faith_txt_del_roll, faith_img_ins_roll, faith_txt_ins_roll, correct_baseline_pred4 = compute_rollout_auc_scores(image_input, text_input, lbl)
    scores_img_del_rand, scores_txt_del_rand, scores_img_ins_rand, scores_txt_ins_rand, auc_img_del_rand, auc_txt_del_rand, auc_img_ins_rand, auc_txt_ins_rand, faith_img_del_rand, faith_txt_del_rand, faith_img_ins_rand, faith_txt_ins_rand, correct_baseline_pred5 = compute_random_baseline(image_input, text_input, lbl, original_idx)

    result_row  = {
        'sample_idx': original_idx,
            'shap': 
                {'scores_img_del_shap': scores_img_del_shap,
                'scores_txt_del_shap': scores_txt_del_shap,
                'scores_img_ins_shap': scores_img_ins_shap,
                'scores_txt_ins_shap': scores_txt_ins_shap,
                'auc_img_del_shap': auc_img_del_shap,
                'auc_txt_del_shap': auc_txt_del_shap,
                'auc_img_ins_shap': auc_img_ins_shap,
                'auc_txt_ins_shap': auc_txt_ins_shap,
                'faith_img_del_shap': faith_img_del_shap,
                'faith_txt_del_shap': faith_txt_del_shap,
                'faith_img_ins_shap': faith_img_ins_shap,
                'faith_txt_ins_shap': faith_txt_ins_shap,
                'correct_baseline_pred': correct_baseline_pred1},
            'rise': 
                {'scores_img_del_rise': scores_img_del_rise,
                'scores_txt_del_rise': scores_txt_del_rise,
                'scores_img_ins_rise': scores_img_ins_rise,
                'scores_txt_ins_rise': scores_txt_ins_rise,
                'auc_img_del_rise': auc_img_del_rise,
                'auc_txt_del_rise': auc_txt_del_rise,
                'auc_img_ins_rise': auc_img_ins_rise,
                'auc_txt_ins_rise': auc_txt_ins_rise,
                'faith_img_del_rise': faith_img_del_rise,
                'faith_txt_del_rise': faith_txt_del_rise,
                'faith_img_ins_rise': faith_img_ins_rise,
                'faith_txt_ins_rise': faith_txt_ins_rise,
                'correct_baseline_pred': correct_baseline_pred2},
            'attention':
                {'scores_img_del_van': scores_img_del_van,
                'scores_txt_del_van': scores_txt_del_van,
                'scores_img_ins_van': scores_img_ins_van,
                'scores_txt_ins_van': scores_txt_ins_van,
                'auc_img_del_van': auc_img_del_van,
                'auc_txt_del_van': auc_txt_del_van,
                'auc_img_ins_van': auc_img_ins_van,
                'auc_txt_ins_van': auc_txt_ins_van,
                'faith_img_del_van': faith_img_del_van,
                'faith_txt_del_van': faith_txt_del_van,
                'faith_img_ins_van': faith_img_ins_van,
                'faith_txt_ins_van': faith_txt_ins_van,
                'correct_baseline_pred': correct_baseline_pred3},
            'rollout':
                {'scores_img_del_roll': scores_img_del_roll,
                'scores_txt_del_roll': scores_txt_del_roll,
                'scores_img_ins_roll': scores_img_ins_roll,
                'scores_txt_ins_roll': scores_txt_ins_roll,
                'auc_img_del_roll': auc_img_del_roll, 
                'auc_txt_del_roll': auc_txt_del_roll,
                'auc_img_ins_roll': auc_img_ins_roll,
                'auc_txt_ins_roll': auc_txt_ins_roll,
                'faith_img_del_roll': faith_img_del_roll,
                'faith_txt_del_roll': faith_txt_del_roll,
                'faith_img_ins_roll': faith_img_ins_roll,
                'faith_txt_ins_roll': faith_txt_ins_roll,
                'correct_baseline_pred': correct_baseline_pred4},
            'random':
                {'scores_img_del_rand': scores_img_del_rand,
                'scores_txt_del_rand': scores_txt_del_rand,
                'scores_img_ins_rand': scores_img_ins_rand,
                'scores_txt_ins_rand': scores_txt_ins_rand,
                'auc_img_del_rand': auc_img_del_rand, 
                'auc_txt_del_rand': auc_txt_del_rand,
                'auc_img_ins_rand': auc_img_ins_rand,
                'auc_txt_ins_rand': auc_txt_ins_rand,
                'faith_img_del_rand': faith_img_del_rand,
                'faith_txt_del_rand': faith_txt_del_rand,
                'faith_img_ins_rand': faith_img_ins_rand,
                'faith_txt_ins_rand': faith_txt_ins_rand,
                'correct_baseline_pred': correct_baseline_pred5}
    }

    # # for both image and text at the same time
    # scores_del_shap, scores_ins_shap, auc_del_shap, auc_ins_shap, spearman_del_img_shap, spearman_ins_img_shap, spearman_del_txt_shap, spearman_ins_txt_shap, correct_baseline_pred1 = compute_shap_auc_scores(image_input, text_input, lbl)
    # scores_del_rise, scores_ins_rise, auc_del_rise, auc_ins_rise, spearman_del_img_rise, spearman_ins_img_rise, spearman_del_txt_rise, spearman_ins_txt_rise, correct_baseline_pred2 = compute_rise_auc_scores(image_input, text_input, lbl)
    # scores_del_van, scores_ins_van, auc_del_van, auc_ins_van, spearman_del_img_van, spearman_ins_img_van, spearman_del_txt_van, spearman_ins_txt_van, correct_baseline_pred3 = get_vanilla_auc_scores(image_input, text_input, lbl)
    # scores_del_roll, scores_ins_roll, auc_del_roll, auc_ins_roll, spearman_del_img_roll, spearman_ins_img_roll, spearman_del_txt_roll, spearman_ins_txt_roll, correct_baseline_pred4 = compute_rollout_auc_scores(image_input, text_input, lbl)
    # scores_del_rand, scores_ins_rand, auc_del_rand, auc_ins_rand, spearman_del_img_rand, spearman_ins_img_rand, spearman_del_txt_rand, spearman_ins_txt_rand, correct_baseline_pred5 = compute_random_baseline(image_input, text_input, lbl, original_idx)

    # result_row = {
    #     'sample_idx': original_idx,
    #         'shap': 
    #             {'scores_del_shap': scores_del_shap,
    #              'scores_ins_shap': scores_ins_shap,
    #              'auc_del_shap': auc_del_shap,
    #              'auc_ins_shap': auc_ins_shap,
    #              'spearman_del_img_shap': spearman_del_img_shap,
    #              'spearman_ins_img_shap': spearman_ins_img_shap,
    #              'spearman_del_txt_shap': spearman_del_txt_shap,
    #              'spearman_ins_txt_shap': spearman_ins_txt_shap,
    #              'correct_baseline_pred': correct_baseline_pred1},
    #         'rise':
    #             {'scores_del_rise': scores_del_rise,
    #              'scores_ins_rise': scores_ins_rise,
    #              'auc_del_rise': auc_del_rise,
    #              'auc_ins_rise': auc_ins_rise,
    #              'spearman_del_img_rise': spearman_del_img_rise,
    #              'spearman_ins_img_rise': spearman_ins_img_rise,
    #              'spearman_del_txt_rise': spearman_del_txt_rise,
    #              'spearman_ins_txt_rise': spearman_ins_txt_rise,
    #              'correct_baseline_pred': correct_baseline_pred2},
    #         'attention':
    #             {'scores_del_van': scores_del_van,
    #              'scores_ins_van': scores_ins_van,
    #              'auc_del_van': auc_del_van,
    #              'auc_ins_van': auc_ins_van,
    #              'spearman_del_img_van': spearman_del_img_van,
    #              'spearman_ins_img_van': spearman_ins_img_van,
    #              'spearman_del_txt_van': spearman_del_txt_van,
    #              'spearman_ins_txt_van': spearman_ins_txt_van,
    #              'correct_baseline_pred': correct_baseline_pred3},
    #         'rollout':
    #             {'scores_del_roll': scores_del_roll,
    #              'scores_ins_roll': scores_ins_roll,
    #              'auc_del_roll': auc_del_roll,
    #              'auc_ins_roll': auc_ins_roll,
    #              'spearman_del_img_roll': spearman_del_img_roll,
    #              'spearman_ins_img_roll': spearman_ins_img_roll,
    #              'spearman_del_txt_roll': spearman_del_txt_roll,
    #              'spearman_ins_txt_roll': spearman_ins_txt_roll,
    #              'correct_baseline_pred': correct_baseline_pred4},
    #         'random':
    #             {'scores_del_rand': scores_del_rand,
    #              'scores_ins_rand': scores_ins_rand,
    #              'auc_del_rand': auc_del_rand,
    #              'auc_ins_rand': auc_ins_rand,
    #              'spearman_del_img_rand': spearman_del_img_rand,
    #              'spearman_ins_img_rand': spearman_ins_img_rand,
    #              'spearman_del_txt_rand': spearman_del_txt_rand,
    #              'spearman_ins_txt_rand': spearman_ins_txt_rand,
    #              'correct_baseline_pred': correct_baseline_pred5}
    # }

    results_df.loc[len(results_df)] = result_row
    ctr += 1

In [ ]:
#results_df.to_json('results_indiv.json', orient='records', lines=True)